In [ ]:
# Cài thư viện tạo nút bấm và giao diện
!pip install ipywidgets

In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# 1. CẤU HÌNH HIỂN THỊ
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

# 2. GIAO DIỆN (UI)
header = widgets.HTML(
    value="<div style='background-color: #2E86C1; color: white; padding: 12px; border-radius: 8px; margin-bottom: 10px;'>"
          "<h2 style='margin: 0;'>🚀 DATA LAKEHOUSE QUERY STUDIO</h2>"
          "<p style='margin: 5px 0 0 0;'>Công cụ truy vấn hệ thống Spark - Iceberg - MinIO</p>"
          "</div>"
)

# Menu chọn kịch bản
dropdown = widgets.Dropdown(
    options=[
        ('--- Chọn kịch bản Demo ---', ''),
        ('1. [HẠ TẦNG] Kiểm tra danh sách bảng (Show Tables)', 'SHOW TABLES IN my_catalog.default'),
        ('2. [SCD TYPE 2] Soi lịch sử thay đổi Khách Hàng', 
         'SELECT customer_id, city, is_current, start_date, end_date FROM my_catalog.default.dim_customers WHERE customer_id IN (SELECT customer_id FROM my_catalog.default.dim_customers WHERE is_current=false LIMIT 1) ORDER BY customer_id, start_date'),
        ('3. [NESTED DATA] Soi cấu trúc Đơn Hàng (Array/Struct)', 
         'SELECT order_id, customer_id, line_items FROM my_catalog.default.fact_orders_nested WHERE size(line_items) > 1 LIMIT 3'),
        ('4. [THỐNG KÊ] Top 5 thành phố đông khách nhất', 
         'SELECT city, count(*) as cnt FROM my_catalog.default.dim_customers WHERE is_current=true GROUP BY city ORDER BY cnt DESC LIMIT 5')
    ],
    value='',
    description='<b>Kịch bản:</b>',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

# Ô NHẬP SQL (Cái này lúc nãy bị thiếu)
sql_area = widgets.Textarea(
    value='',
    placeholder='Câu lệnh SQL sẽ hiện ở đây khi em chọn kịch bản. Hoặc em có thể tự gõ lệnh...',
    description='',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='98%', height='100px')
)

# Nút chạy
btn_run = widgets.Button(
    description=' CHẠY TRUY VẤN',
    button_style='primary', # Màu xanh dương
    icon='play',
    layout=widgets.Layout(width='200px', height='40px', margin='10px 0px')
)

# Vùng hiển thị kết quả
out = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '15px', 'margin-top': '10px'})

# 3. XỬ LÝ (BACKEND)
def on_dropdown_change(change):
    # Khi chọn menu thì tự động điền code vào ô SQL
    if change['new']:
        sql_area.value = change['new']

def on_click(b):
    with out:
        clear_output()
        # Lấy code từ ô nhập liệu (sql_area) thay vì dropdown
        sql = sql_area.value.strip()
        
        if not sql:
            print("⚠️ Vui lòng chọn kịch bản hoặc nhập code SQL!")
            return

        print(f"⏳ Đang chạy lệnh: {sql} ...")
        try:
            df = spark.sql(sql).toPandas()
            print(f"✅ Tìm thấy {len(df)} dòng dữ liệu.\n")
            if not df.empty:
                display(df.style.set_table_styles([
                    {'selector': 'th', 'props': [('background-color', '#2E86C1'), ('color', 'white'), ('text-align', 'center')]},
                    {'selector': 'td', 'props': [('border', '1px solid #ddd'), ('padding', '8px')]},
                    {'selector': 'tr:nth-of-type(even)', 'props': [('background-color', '#f2f2f2')]}
                ]))
            else:
                print("⚠️ Bảng rỗng.")
        except Exception as e:
            print(f"❌ LỖI: {e}")

# Gắn sự kiện
dropdown.observe(on_dropdown_change, names='value')
btn_run.on_click(on_click)

# 4. HIỂN THỊ (Đã thêm sql_area vào)
display(header, dropdown, sql_area, btn_run, out)

HTML(value="<div style='background-color: #2E86C1; color: white; padding: 12px; border-radius: 8px; margin-bot…

Dropdown(description='<b>Kịch bản:</b>', layout=Layout(width='80%'), options=(('--- Chọn kịch bản Demo ---', '…

Textarea(value='', layout=Layout(height='100px', width='98%'), placeholder='Câu lệnh SQL sẽ hiện ở đây khi em …

Button(button_style='primary', description=' CHẠY TRUY VẤN', icon='play', layout=Layout(height='40px', margin=…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…